In [1]:
!pip install -q \
numpy \
pandas \
scipy \
scikit-learn \
catboost \
lightgbm \
tqdm


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis
from scipy.signal import welch

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import ExtraTreesClassifier

from catboost import CatBoostClassifier

from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)

In [3]:
DATA_DIR = "/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment"

TRAIN_ACCEL = f"{DATA_DIR}/train-accel.csv"
TRAIN_GYRO  = f"{DATA_DIR}/train-gyro.csv"
TRAIN_LABEL = f"{DATA_DIR}/train-label.csv"

TEST_ACCEL = f"{DATA_DIR}/test-accel.csv"
TEST_GYRO  = f"{DATA_DIR}/test-gyro.csv"
TEST_LABEL = f"{DATA_DIR}/test-label.csv"

In [4]:
train_accel = pd.read_csv(TRAIN_ACCEL)
train_gyro  = pd.read_csv(TRAIN_GYRO)
train_label = pd.read_csv(TRAIN_LABEL)

test_accel = pd.read_csv(TEST_ACCEL)
test_gyro  = pd.read_csv(TEST_GYRO)
test_label = pd.read_csv(TEST_LABEL)

print(train_accel.shape)
print(train_gyro.shape)
print(train_label.shape)

print(test_accel.shape)
print(test_gyro.shape)
print(test_label.shape)

(2428374, 7)
(2433673, 7)
(38015, 4)
(2528310, 7)
(2541831, 7)
(39473, 4)


In [5]:
def add_device_features(df):

    df = df.copy()

    df["device"] = df["device"].fillna("unknown")

    df["device_is_apple"] = (
        df["device"].str.lower()
        .str.contains("apple")
        .astype(int)
    )

    df["device_is_unknown"] = (
        df["device"].str.lower()
        .eq("unknown")
        .astype(int)
    )

    return df


train_accel = add_device_features(train_accel)
train_gyro  = add_device_features(train_gyro)

test_accel = add_device_features(test_accel)
test_gyro  = add_device_features(test_gyro)

In [6]:
def correct_direction(df):

    df = df.copy()

    xb = np.zeros(len(df))
    yb = np.zeros(len(df))
    zb = df["z"].values

    x = df["x"].values
    y = df["y"].values

    d = df["direction"].values

    idx = d == 1
    xb[idx] = x[idx]
    yb[idx] = y[idx]

    idx = d == 2
    xb[idx] = y[idx]
    yb[idx] = -x[idx]

    idx = d == 3
    xb[idx] = -x[idx]
    yb[idx] = -y[idx]

    idx = d == 4
    xb[idx] = -y[idx]
    yb[idx] = x[idx]

    df["xb"] = xb
    df["yb"] = yb
    df["zb"] = zb

    return df


train_accel = correct_direction(train_accel)
train_gyro  = correct_direction(train_gyro)

test_accel = correct_direction(test_accel)
test_gyro  = correct_direction(test_gyro)

In [7]:
for df in [train_accel, test_accel]:

    df["amag"] = np.sqrt(
        df["xb"]**2 +
        df["yb"]**2 +
        df["zb"]**2
    )

for df in [train_gyro, test_gyro]:

    df["gmag"] = np.sqrt(
        df["xb"]**2 +
        df["yb"]**2 +
        df["zb"]**2
    )

In [8]:
def dominant_frequency(x, fs=50):

    if len(x) < 20:
        return 0

    try:

        freq, power = welch(
            x,
            fs=fs,
            nperseg=min(len(x), 128)
        )

        return freq[np.argmax(power)]

    except:
        return 0

In [9]:
def autocorr_lag(x, lag):

    if len(x) <= lag:
        return 0

    a = x[:-lag]
    b = x[lag:]

    if np.std(a) == 0 or np.std(b) == 0:
        return 0

    return np.corrcoef(a, b)[0, 1]

In [10]:
def build_features(accel, gyro):

    accel = accel.copy()
    gyro = gyro.copy()

    accel["sec"] = accel["time"].astype(int)
    gyro["sec"] = gyro["time"].astype(int)

    keys = ["pid", "sec"]

    rows = []

    accel_groups = accel.groupby(keys)
    gyro_groups = gyro.groupby(keys)

    common_keys = sorted(
        set(accel_groups.groups.keys())
        &
        set(gyro_groups.groups.keys())
    )

    for key in tqdm(common_keys):

        a = accel_groups.get_group(key)
        g = gyro_groups.get_group(key)

        feat = {
            "pid": key[0],
            "time": key[1]
        }

        feat["direction"] = a["direction"].iloc[0]
        feat["device_is_apple"] = a["device_is_apple"].iloc[0]
        feat["device_is_unknown"] = a["device_is_unknown"].iloc[0]

        for col in ["xb","yb","zb","amag"]:

            x = a[col].values

            feat[f"{col}_mean"] = np.mean(x)
            feat[f"{col}_std"] = np.std(x)
            feat[f"{col}_min"] = np.min(x)
            feat[f"{col}_max"] = np.max(x)

            feat[f"{col}_skew"] = skew(x)
            feat[f"{col}_kurt"] = kurtosis(x)

            feat[f"{col}_fft"] = dominant_frequency(x)

            feat[f"{col}_ac1"] = autocorr_lag(x, 25)
            feat[f"{col}_ac2"] = autocorr_lag(x, 50)

        for col in ["xb","yb","zb","gmag"]:

            x = g[col].values

            feat[f"g_{col}_mean"] = np.mean(x)
            feat[f"g_{col}_std"] = np.std(x)
            feat[f"g_{col}_max"] = np.max(x)

            feat[f"g_{col}_fft"] = dominant_frequency(x)

        rows.append(feat)

    return pd.DataFrame(rows)

In [11]:
train_feat = build_features(
    train_accel,
    train_gyro
)

test_feat = build_features(
    test_accel,
    test_gyro
)

print(train_feat.shape)
print(test_feat.shape)

100%|██████████| 50715/50715 [02:57<00:00, 285.86it/s]


(47893, 57)
(50715, 57)


In [12]:
train_df = train_feat.merge(
    train_label,
    on=["pid", "time"],
    how="inner"
)

test_df = test_feat.merge(
    test_label,
    on=["pid", "time"],
    how="inner"
)

print(train_df.shape)
print(test_df.shape)

(30896, 59)
(30707, 59)


In [13]:
base_cols = [
    c for c in train_df.columns
    if c not in ["pid","id","time","workout"]
]

for col in tqdm(base_cols):

    train_df[f"{col}_roll3"] = (
        train_df
        .groupby("pid")[col]
        .transform(
            lambda s:
            s.rolling(
                7,
                center=True,
                min_periods=1
            ).mean()
        )
    )

    test_df[f"{col}_roll3"] = (
        test_df
        .groupby("pid")[col]
        .transform(
            lambda s:
            s.rolling(
                7,
                center=True,
                min_periods=1
            ).mean()
        )
    )

100%|██████████| 55/55 [00:00<00:00, 79.43it/s]


In [14]:
FEATURES = [
    c for c in train_df.columns
    if c not in [
        "id",
        "pid",
        "time",
        "workout"
    ]
]

X = train_df[FEATURES]
y = train_df["workout"]

X_test = test_df[FEATURES]

groups = train_df["pid"]

In [15]:
gkf = GroupKFold(n_splits=5)

oof = np.zeros(len(X))

test_pred = np.zeros(
    (len(X_test), 9)
)

scores = []

for fold, (tr, va) in enumerate(
    gkf.split(X, y, groups)
):

    print(f"\nFold {fold+1}")

    model = CatBoostClassifier(
        iterations=1200,
        depth=8,
        learning_rate=0.03,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        verbose=0,
        random_seed=42
    )

    model.fit(
        X.iloc[tr],
        y.iloc[tr]
    )

    pred = model.predict(
        X.iloc[va]
    ).astype(int).ravel()

    score = balanced_accuracy_score(
        y.iloc[va],
        pred
    )

    scores.append(score)

    print(score)

    oof[va] = pred

    test_pred += (
        model.predict_proba(X_test) / 5
    )


Fold 1
0.8433063725520911

Fold 2
0.8300099826826896

Fold 3
0.8597539126679732

Fold 4
0.8105463782562888

Fold 5
0.844512988862546


In [16]:
print("\nFold scores")

for s in scores:
    print(round(s,6))

print()

print("Mean:", np.mean(scores))
print("Std :", np.std(scores))

print(
    "Variance penalized:",
    np.mean(scores) -
    0.5*np.std(scores)
)


Fold scores
0.843306
0.83001
0.859754
0.810546
0.844513

Mean: 0.8376259270043178
Std : 0.01649637172531121
Variance penalized: 0.8293777411416622


In [17]:
test_df["pred"] = np.argmax(
    test_pred,
    axis=1
)

test_df = test_df.sort_values(
    ["pid","time"]
)

test_df["pred"] = (
    test_df
    .groupby("pid")["pred"]
    .transform(
        lambda s:
        s.rolling(
            3,
            center=True,
            min_periods=1
        )
        .apply(
            lambda x:
            pd.Series(x).mode()[0]
        )
    )
)

test_df["pred"] = (
    test_df["pred"]
    .astype(int)
)

In [18]:
submission = test_label[["id"]].copy()

submission["workout"] = (
    test_df
    .sort_values("id")
    ["pred"]
    .values
)

submission.to_csv(
    "submission_p10.csv",
    index=False
)

submission.head()

ValueError: Length of values (30707) does not match length of index (39473)